### Evaluation of RAG pipline/ Testing the RAG

* Chose which llm is best in producing the result.
* we need to test our rag app whether it produce correct output or not.

#### step 1: create datsets ( questions and answer)

In [32]:
import os
from dotenv import load_dotenv
load_dotenv()

from langsmith import Client
client= Client()

dataset_name = "rag_evaluation"
dataset = client.create_dataset(dataset_name)

## Create datasets...
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "explain about leave policey"},
            "outputs": {"answer": "Zen X’s leave policy grants 15 earned days and 10 sick days per year, credited monthly, and requires all requests to be submitted via the HR portal or to a manager at least two days in advance. Leave is unapproved until manager approval; unplanned or emergency leave is handled under the emergency procedures, with loss of pay or non‑compliance penalties applied if policies are breached"}
        },
        {
            "inputs": {"question": "How many days Casual Leave can be taken"},
            "outputs": {"answer": " 10 days per calendar year"}
        }
    ]
)


{'example_ids': ['cd39fcc0-2361-4add-8912-173bb3c87e6b',
  '0a286190-ece4-4e42-876d-6421ec50dc17'],
 'count': 2}

### Define evaluation metrics

In [ ]:
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
from langsmith import wrappers
from groq import Groq

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
model = "openai/gpt-oss-20b"

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""
    you are grading the following question:
    {inputs["question"]}

    here is the real answer:
    {reference_outputs["answer"]}

    you are grading the following predicted answer:
    {outputs["response"]}

    Respond with CORRECT or INCORRECT:
    """

    response = groq_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are an expert professor specialized in grading students"},
            {"role": "user", "content": user_content}
        ]
    ).choices[0].message.content.strip()
    print(response)
    return response == "CORRECT"



## Concisions- checks whether the actual output is less than 2x the length of the expected result.
def concision(outputs: dict, reference_outputs: dict) -> bool:
    return len(outputs["response"]) < 2 * len(reference_outputs["answer"])



In [40]:
from agentic_rag import agentic_app
from langsmith.evaluation import evaluate

def ls_target(input: str)->dict:
    sample_question = {
        "messages": [{"role": "user", "content": input["question"]}],
        "question": input["question"],
        "retry_count": 0,
        "feedback": "notthing for now",
        "is_answer_good": True
    }
    response = agentic_app.invoke(sample_question)
    print("response from llm is:::")
    print(response["answer"])
    return {"response": response["answer"]}

evaluation_results = evaluate(
    ls_target,                    
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="new_model"
)


View the evaluation results for experiment: 'new_model-ffcd4689' at:
https://smith.langchain.com/o/00eef8fd-2e40-4510-a4c0-c96c3faec2f7/datasets/1f0f326a-443a-48ae-b7b8-b3afd7fda035/compare?selectedSessions=2314f70f-24f2-4595-ae37-5bafc3a658ac




0it [00:00, ?it/s]

Sub questions are
Identify the leave policy source, locate the Casual Leave entitlement clause, calculate the total number of days allowed.
response from llm is:::
According to Company Zen X’s leave policy, employees are entitled to **10 days of Casual Leave (CL) per calendar year**. This is the maximum number of CL days that can be taken within any given year.


1it [00:01,  1.69s/it]

CORRECT
Sub questions are
Define what a leave policy is, list common types of leave, explain eligibility and application process
response from llm is:::
Zen X’s leave policy treats all absences as **unapproved** until a manager gives permission in the HR portal, otherwise they become **Loss of Pay**. Employees can take up to 10 days of **Casual Leave** and 10 days of **Sick Leave** per calendar year, and must apply through the portal or via manager email at least 2 days in advance. Unplanned leave is handled similarly, with Comp‑Off required to be used within 30 days of approval.


2it [00:03,  1.95s/it]

INCORRECT


2it [00:04,  2.22s/it]
